# Correlação de Pearson — base A601

Análise descritiva das seis entradas da MLP e da ETo calculada por FAO-56. Todos os coeficientes usam a mesma população de casos completos; não há exclusão diferente por par nem imputação.

Execute em kernel novo, na ordem, a partir da raiz do projeto ou de `matriz_correlacao/`. As saídas ficam em `matriz_correlacao/resultados/` e são sobrescritas ao executar novamente. Consulte o [README](README.md) e a [documentação dos dados](../FAO-56/README.md).

In [ ]:
from datetime import datetime, timezone
import csv
import hashlib
from importlib.metadata import version
import json
from pathlib import Path
import platform

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


## 1. Configuração

Os caminhos são resolvidos tanto da raiz quanto da pasta deste notebook. Os nomes e a ordem das colunas correspondem a entrada da MLP. O CSV mantém a precisão dos coeficientes; somente a figura arredonda para duas casas decimais.

In [ ]:
def localizar_raiz(diretorio):
    diretorio = Path(diretorio).resolve()
    for candidata in (diretorio, diretorio.parent):
        if (candidata / 'mlp_FAO-56.ipynb').is_file() and (candidata / 'FAO-56').is_dir():
            return candidata
    raise FileNotFoundError('Execute na raiz do projeto ou em matriz_correlacao/.')

ROOT = localizar_raiz(Path.cwd())
ARQUIVO = ROOT / 'FAO-56' / 'dados_FAO-56' / 'dataset_A601_mlp.csv'
PASTA_SAIDA = ROOT / 'matriz_correlacao' / 'resultados'
EXPORTAR_TABELA_ETO = False  # subconjunto da matriz; exportação opcional
COLUNAS = [
    'rn_mj_m2_dia', 'u2_ms', 'ur_media_pct',
    'tmax_c', 'tmin_c', 'tmedia_inmet_c', 'eto_pmt',
]
ROTULOS = ['Rn', 'u2', 'UR', 'Tmax', 'Tmin', 'Tmed', 'ETo']
print(f'Entrada: {ARQUIVO}')
print(f'Saída: {PASTA_SAIDA}')


## 2. Leitura e validação

Datas ausentes ou repetidas, colunas ausentes ou duplicadas, valores não numéricos e infinitos interrompem a análise. Valores inválidos não são convertidos silenciosamente em NaN. Linhas com ausências nas sete variáveis são removidas em conjunto, como na MLP. A quantidade removida e o período utilizado são registrados; variáveis constantes impedem o cálculo de Pearson.

In [ ]:
def carregar_dados(caminho, colunas):
    with Path(caminho).open(encoding='utf-8-sig', newline='') as arquivo:
        cabecalho = next(csv.reader(arquivo), [])
    if len(cabecalho) != len(set(cabecalho)):
        raise ValueError('O CSV contém nomes de colunas duplicados.')
    tabela = pd.read_csv(caminho, encoding='utf-8-sig')
    faltantes = sorted(set(['data', *colunas]) - set(tabela.columns))
    if faltantes:
        raise ValueError(f'Colunas obrigatórias ausentes: {faltantes}')
    datas = pd.to_datetime(tabela['data'], format='%Y-%m-%d', errors='raise')
    if datas.isna().any() or datas.duplicated().any():
        raise ValueError('Datas ausentes ou duplicadas.')
    numericos = tabela[colunas].copy()
    for coluna in colunas:
        numericos[coluna] = pd.to_numeric(numericos[coluna], errors='raise')
    if np.isinf(numericos.to_numpy(dtype=float)).any():
        raise ValueError('Existem valores infinitos no dataset.')
    completos = numericos.notna().all(axis=1)
    dados = numericos.loc[completos].copy()
    if len(dados) < 3:
        raise ValueError('São necessárias pelo menos três observações completas.')
    constantes = dados.columns[dados.nunique() < 2].tolist()
    if constantes:
        raise ValueError(f'Pearson indefinido para variáveis constantes: {constantes}')
    resumo = {
        'linhas_entrada': len(tabela),
        'linhas_utilizadas': len(dados),
        'linhas_removidas': int((~completos).sum()),
        'ausencias_por_coluna': numericos.isna().sum().astype(int).to_dict(),
        'data_inicial_utilizada': datas.loc[completos].min().strftime('%Y-%m-%d'),
        'data_final_utilizada': datas.loc[completos].max().strftime('%Y-%m-%d'),
    }
    return dados, resumo

# Hash antes e depois da leitura para detectar alteração do arquivo nesta etapa.
hash_antes = hashlib.sha256(ARQUIVO.read_bytes()).hexdigest()
dados, resumo_dados = carregar_dados(ARQUIVO, COLUNAS)
dataset_sha256 = hashlib.sha256(ARQUIVO.read_bytes()).hexdigest()
if dataset_sha256 != hash_antes:
    raise RuntimeError('O dataset mudou durante a leitura; execute novamente.')
print(json.dumps(resumo_dados, ensure_ascii=False, indent=2))


## 3. Matriz e correlações com ETo

Pearson mede associação linear e varia de −1 a 1. A matriz é calculada nas unidades originais, sem normalização. A tabela de ETo é apenas uma visão da matriz, ordenada pelo valor absoluto de r; ela não seleciona automaticamente entradas para a MLP.

In [ ]:
matriz_corr = dados.corr(method='pearson')
valores = matriz_corr.to_numpy()
if not np.isfinite(valores).all() or (np.abs(valores) > 1 + 1e-12).any():
    raise ValueError('Coeficientes de Pearson não finitos ou fora de [-1, 1].')
np.testing.assert_allclose(valores, valores.T, rtol=0, atol=1e-12)
np.testing.assert_allclose(np.diag(valores), 1, rtol=0, atol=1e-12)

correlacao_eto = (
    matriz_corr['eto_pmt'].drop('eto_pmt').rename('Pearson_r')
    .rename_axis('Variavel').reset_index()
)
correlacao_eto['Pearson_abs'] = correlacao_eto['Pearson_r'].abs()
correlacao_eto = correlacao_eto.sort_values(
    'Pearson_abs', ascending=False, kind='stable',
).reset_index(drop=True)
print(matriz_corr.round(4).to_string())
print('\nCorrelações com ETo:')
print(correlacao_eto.to_string(index=False))


## 4. Figura e exportação

A figura usa escala divergente fixa de −1 a 1, centrada em zero, rótulos curtos e número de observações. A matriz CSV é relida e comparada com a calculada. O JSON registra hash do dataset, população, opções e versões. Os nomes fixos permitem que a MLP encontre a matriz após cada execução; preserve a pasta separadamente se precisar guardar versões anteriores.

In [ ]:
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
caminho_csv = PASTA_SAIDA / 'matriz_correlacao_pearson.csv'
matriz_corr.to_csv(caminho_csv, encoding='utf-8-sig', index_label='variavel')
lida = pd.read_csv(caminho_csv, index_col=0, float_precision='round_trip')
lida.index.name = None
pd.testing.assert_frame_equal(lida, matriz_corr, check_exact=False, rtol=1e-12, atol=1e-12)

with plt.rc_context({'font.size': 11}):
    fig, ax = plt.subplots(figsize=(8, 7), layout='constrained')
    imagem = ax.imshow(valores, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xticks(np.arange(len(COLUNAS)), labels=ROTULOS)
    ax.set_yticks(np.arange(len(COLUNAS)), labels=ROTULOS)
    ax.set_title(f'Correlação de Pearson — INMET A601\nCasos completos: n = {len(dados):,}')
    for i in range(len(COLUNAS)):
        for j in range(len(COLUNAS)):
            valor = valores[i, j]
            ax.text(j, i, f'{valor:.2f}', ha='center', va='center',
                    color='white' if abs(valor) >= 0.6 else 'black')
    fig.colorbar(imagem, ax=ax, shrink=0.85, label='Coeficiente de Pearson (r)')
    fig.savefig(PASTA_SAIDA / 'matriz_correlacao_pearson.png', dpi=300)
    plt.show()
    plt.close(fig)

arquivos = ['matriz_correlacao_pearson.csv', 'matriz_correlacao_pearson.png']
if EXPORTAR_TABELA_ETO:
    caminho_eto = PASTA_SAIDA / 'correlacao_com_ETo.csv'
    correlacao_eto.to_csv(caminho_eto, index=False, encoding='utf-8-sig')
    pd.testing.assert_frame_equal(pd.read_csv(caminho_eto), correlacao_eto,
                                  check_exact=False, rtol=1e-12, atol=1e-12)
    arquivos.append(caminho_eto.name)

registro = {
    'metodo': 'pearson',
    'escopo': 'descritivo; população completa anterior ao split da MLP',
    'politica_ausencias': 'casos completos nas sete variáveis; sem imputação',
    'dataset': str(ARQUIVO.relative_to(ROOT)),
    'dataset_sha256': dataset_sha256,
    'colunas': COLUNAS,
    'populacao': resumo_dados,
    'versoes': {'python': platform.python_version(), **{
        pacote: version(pacote) for pacote in ['numpy', 'pandas', 'matplotlib']
    }},
    'exportar_tabela_eto': EXPORTAR_TABELA_ETO,
    'arquivos_gerados': arquivos,
    'fim_utc': datetime.now(timezone.utc).isoformat(),
    'status': 'concluida',
}
(PASTA_SAIDA / 'configuracao_correlacao.json').write_text(
    json.dumps(registro, ensure_ascii=False, indent=2, allow_nan=False) + '\n',
    encoding='utf-8',
)
print(f'Exportação concluída em {PASTA_SAIDA}')


## 5. Limites da interpretação

- Os coeficientes descrevem esta população completa, incluindo dias que a MLP posteriormente usa em validação e teste. Não são uma etapa de seleção de variáveis ajustada somente no treino.
- ETo é um alvo calculado. Rn e outras entradas participam direta ou indiretamente de sua construção; correlações altas não demonstram validação independente da evapotranspiração.
- Pearson mede associação linear, não causalidade nem desempenho preditivo de uma MLP.
- A série diária pode apresentar sazonalidade e dependência temporal. Não são calculados p-valores ou intervalos de confiança com hipótese de observações independentes.
- As limitações de qualidade da estação e do processamento estão no [README FAO-56](../FAO-56/README.md).